In [2]:
from langchain.agents import initialize_agent, Tool, load_tools
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.tools import BaseTool
from langchain.memory import ConversationBufferMemory
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import Chroma
import pandas as pd
import fitz

import re
import json
import os
from dotenv import load_dotenv

In [3]:
dotenv_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
load_dotenv(dotenv_path)

API_KEY = os.getenv("GOOGLE_API_KEY")

In [4]:
# --- 1. Load and prepare CSV data for RAG ---
# Assuming your CSV is named 'my_data.csv'
loader = CSVLoader(file_path='data/processed/openlca_results_recorrected_calcsetup_ELCD.csv')
documents = loader.load()

# Initialize embeddings model (Gemini embedding models are usually good)
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001") # Use an appropriate embedding model

# Create a vector store (e.g., Chroma for simplicity)
vectorstore = Chroma.from_documents(documents, embeddings)

# Create a retriever for the vector store
retriever = vectorstore.as_retriever()

In [5]:
class CSV_query(BaseTool):
    name : str = "csv_query"
    description : str = "Useful for answering questions about the data in the CSV.Input should be a question or keywords related to the CSV content.Returns relevant information from the CSV."

    def _run(self, query: str):
        relevant_docs = retriever.invoke(query)
        # You might want to format the docs nicely or summarize them here
        return "\n".join([doc.page_content for doc in relevant_docs])

In [6]:
# Tool para leer un PDF
class LeerPDFTool(BaseTool):
    name : str = "leer_pdf"
    description : str = "Lee información de un PDF sobre un producto"

    def _run(self, file_path: str):
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    


# # Tool para consultar CSVs con factores de emisión
# class ConsultarCSVTool(BaseTool):
#     name : str = "consultar_csv"
#     description : str = "Consulta factores de emisión en archivos CSV"

#     def _run(self, query: str):
#         procesos = pd.read_csv("data/processed/openlca_results_recorrected_calcsetup_ELCD.csv")   #### DEFINIR BIEN LOS CSVs ####
#         return {
#             "procesos": procesos.to_dict(orient='records'),
#         }
    
 

# Tool para calcular la huella de carbono (ejemplo simple)
class CalculadoraEmisionesTool(BaseTool):
    name : str = "calcular_emisiones"
    description : str = "Calcula las emisiones estimadas usando datos del producto"

    def _run(self, info_producto: str):
        # Aquí haces el parseo del texto y haces cálculos con pandas
        # Ejemplo: parsear que hay 2kg de acero -> buscar en CSV cuánto CO₂ por kg
        return "Emisión estimada: 3.4 kg CO₂e"
    



In [7]:
created_tools = [
    LeerPDFTool(),
    CSV_query(),
    # ConsultarCSVTool(),
    CalculadoraEmisionesTool()
]

In [8]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite", google_api_key=API_KEY, temperature=0)
tools = load_tools(['llm-math'], llm=llm)


agent = initialize_agent(
    tools + created_tools,
    llm,
    agent_type="zero-shot-react-description",
    memory=ConversationBufferMemory(memory_key="chat_history"),
    verbose=True,
)

C:\Users\danie\AppData\Local\Temp\ipykernel_2900\3655095810.py:9: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(memory_key="chat_history"),
C:\Users\danie\AppData\Local\Temp\ipykernel_2900\3655095810.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [9]:
FILE = 'temp-files/temp.pdf'
# CSV = 'data/processed/openlca_results_recorrected_calcsetup_ELCD.csv'
from data.resumen_muestra import resumen as RESUMEN_GENERAL

In [ ]:
agent_prompt = f"""
Eres un analista de productos especializado en calcular la huella de carbono de un producto, tu objetivo es utilizar los datos de un archivo pdf de un producto y calcular cual es su huella estimada.

Tendras que:
1. Clacular la huella de carbono de cada uno de los procesos [Materias Primas, Fabricacion, Packaging, Transporte, Uso, Fin de Vida]. Para esto vas a hacer uso de la herramienta "csv_query"
2. Calcular la huella de carbono total haciendo una suma de todos los proceso
3. Estimar la media de la industria y productos similares y calcualren porcientos la diferencia de este producto en relacion a la media (si esta por debajo -x%, si esta por encima x%)
4. Resumen general de las conclusiones sacadas del proceso imitando el formato que te doy. EL resumen debe ser en ingles

Resumen: {RESUMEN_GENERAL}

Para esto consultaras el CSV que tienes como herramienta donde tienes una lista de procesos y su emision de carbono. Compararas los procesos del producto y estimaras la emision de cada proceso.
Al final calcularas el total de emision de CO2 del producto
Esta bien que hagas asunciones si ves que te faltan datos, intenta que estas asunciones sean lo mas precisas posibles
PDF: {FILE}

La respuesta se usara para meter los datos en una base de datos asi que debe ser ÚNICAMENTE el siguiente objeto JSON, sin ningún texto antes o después, ni explicaciones, ni comentarios:

    {{
    "output": {{
            "emision_materia_prima": "...",
            "emision_fabricacion": "...",
            "emision_packaging" : "...",
            "emision_transporte" : "...",
            "emision_uso" : "...",
            "emision_fin_de_vida" : "...",
            "emision_total" : "...",
            "diferencia_porcentual" : "...",
            "resumen_general": "..."
            }}

    }}


El JSON devuelvelo sin saltos de linea, lo necescito asi

"""

In [11]:
data = agent.invoke(agent_prompt)['output']



> Entering new AgentExecutor chain...
I need to analyze a product's carbon footprint using a PDF and a CSV. I need to extract emissions data for different processes, calculate the total emissions, and provide a summary in a specific JSON format. I will start by reading the PDF to understand the product and its processes.
Action: leer_pdf
Action Input: temp-files/temp.pdf
Observation: 💧 AGUA DANONE (Botella PET 1,5L) 
Materias Primas:​
El producto se compone exclusivamente de agua mineral natural extraída de manantiales 
como Evian o Volvic, situados en Francia. La captación se realiza directamente desde 
acuíferos subterráneos mediante sistemas de bombeo sellados que evitan la contaminación 
del recurso. 
Fabricación:​
El agua se somete a controles de calidad y filtrado antes de ser embotellada. El proceso de 
llenado se realiza en condiciones higiénicas estrictas y automáticamente. La planta de 
embotellado de Evian, por ejemplo, funciona con un 100% de energía renovable. 
Packaging

In [12]:
data

'```json\n{\n    "output": {\n            "emision_materia_prima": "0.1",\n            "emision_fabricacion": "0.05",\n            "emision_packaging" : "0.5",\n            "emision_transporte" : "0.2",\n            "emision_uso" : "0",\n            "emision_fin_de_vida" : "-0.1",\n            "resumen_general": "General Conclusion: The Life Cycle Assessment (LCA) of Danone water demonstrates a balanced approach to sustainability. The product excels in the use of renewable resources and recycled materials, yet there are opportunities for further improvements in ingredient selection and end-of-life material management.\\nStrengths:\\nEnergy efficiency in manufacturing: The product benefits from the use of 100% renewable energy in the bottling plant. \\nUse of recycled materials in packaging: rPET used in packaging is recycled, significantly reducing carbon emissions and advancing circular economy principles.\\nOptimized transportation: Distribution uses trucks and trains for long routes

In [13]:
json_match = re.search(r'\{.*\}', data, re.DOTALL)
if json_match:
    data = json_match.group(0)

data = json.loads(data)

In [14]:
data

{'output': {'emision_materia_prima': '0.1',
  'emision_fabricacion': '0.05',
  'emision_packaging': '0.5',
  'emision_transporte': '0.2',
  'emision_uso': '0',
  'emision_fin_de_vida': '-0.1',
  'resumen_general': 'General Conclusion: The Life Cycle Assessment (LCA) of Danone water demonstrates a balanced approach to sustainability. The product excels in the use of renewable resources and recycled materials, yet there are opportunities for further improvements in ingredient selection and end-of-life material management.\nStrengths:\nEnergy efficiency in manufacturing: The product benefits from the use of 100% renewable energy in the bottling plant. \nUse of recycled materials in packaging: rPET used in packaging is recycled, significantly reducing carbon emissions and advancing circular economy principles.\nOptimized transportation: Distribution uses trucks and trains for long routes. \nMinimal impact during use phase: The product does not consume energy during use, eliminating this ph

In [17]:
data['output']['resumen_general']

'General Conclusion: The Life Cycle Assessment (LCA) of Danone water demonstrates a balanced approach to sustainability. The product excels in the use of renewable resources and recycled materials, yet there are opportunities for further improvements in ingredient selection and end-of-life material management.\nStrengths:\nEnergy efficiency in manufacturing: The product benefits from the use of 100% renewable energy in the bottling plant. \nUse of recycled materials in packaging: rPET used in packaging is recycled, significantly reducing carbon emissions and advancing circular economy principles.\nOptimized transportation: Distribution uses trucks and trains for long routes. \nMinimal impact during use phase: The product does not consume energy during use, eliminating this phase as a contributor to its total footprint.\nPromising end-of-life management: Recycling rates reach up to 60% of the plastic bottles.\nPotential areas for Improvement:\nIncrease recycled content: Enhancing recycl